In [9]:
# extracting api

import requests


def extract_binance():

    url = f"https://api.binance.com/api/v3/ticker/24hr"

    params = {
        'symbolStatus': 'TRADING',
        'type': 'FULL'
    }

    response = requests.get(url,params=params)


    response.raise_for_status()

    data = response.json()

    return data

binance_data = extract_binance()

print(binance_data)

[{'symbol': 'ETHBTC', 'priceChange': '-0.00020000', 'priceChangePercent': '-0.642', 'weightedAvgPrice': '0.03133544', 'prevClosePrice': '0.03114000', 'lastPrice': '0.03094000', 'lastQty': '0.25070000', 'bidPrice': '0.03094000', 'bidQty': '47.86030000', 'askPrice': '0.03095000', 'askQty': '7.67730000', 'openPrice': '0.03114000', 'highPrice': '0.03158000', 'lowPrice': '0.03090000', 'volume': '21413.97630000', 'quoteVolume': '671.01634725', 'openTime': 1776425986866, 'closeTime': 1776512386866, 'firstId': 528117319, 'lastId': 528175196, 'count': 57878}, {'symbol': 'LTCBTC', 'priceChange': '-0.00000800', 'priceChangePercent': '-1.075', 'weightedAvgPrice': '0.00073738', 'prevClosePrice': '0.00074400', 'lastPrice': '0.00073600', 'lastQty': '47.14200000', 'bidPrice': '0.00073600', 'bidQty': '232.13300000', 'askPrice': '0.00073700', 'askQty': '383.68900000', 'openPrice': '0.00074400', 'highPrice': '0.00074800', 'lowPrice': '0.00072600', 'volume': '7935.16500000', 'quoteVolume': '5.85122718', '

In [10]:
binance_data

[{'symbol': 'ETHBTC',
  'priceChange': '-0.00020000',
  'priceChangePercent': '-0.642',
  'weightedAvgPrice': '0.03133544',
  'prevClosePrice': '0.03114000',
  'lastPrice': '0.03094000',
  'lastQty': '0.25070000',
  'bidPrice': '0.03094000',
  'bidQty': '47.86030000',
  'askPrice': '0.03095000',
  'askQty': '7.67730000',
  'openPrice': '0.03114000',
  'highPrice': '0.03158000',
  'lowPrice': '0.03090000',
  'volume': '21413.97630000',
  'quoteVolume': '671.01634725',
  'openTime': 1776425986866,
  'closeTime': 1776512386866,
  'firstId': 528117319,
  'lastId': 528175196,
  'count': 57878},
 {'symbol': 'LTCBTC',
  'priceChange': '-0.00000800',
  'priceChangePercent': '-1.075',
  'weightedAvgPrice': '0.00073738',
  'prevClosePrice': '0.00074400',
  'lastPrice': '0.00073600',
  'lastQty': '47.14200000',
  'bidPrice': '0.00073600',
  'bidQty': '232.13300000',
  'askPrice': '0.00073700',
  'askQty': '383.68900000',
  'openPrice': '0.00074400',
  'highPrice': '0.00074800',
  'lowPrice': '0.0

In [ ]:
# transforming data

import pandas as pd
from extract import binance_data
from datetime import datetime

def transform_binance():

    df = pd.DataFrame(binance_data)

    cols_to_keep = [
        'symbol', 'priceChange', 'priceChangePercent', 
        'lastPrice', 'volume', 'quoteVolume', 'openTime'
    ]

    df = df[cols_to_keep]

    numeric_cols = ['priceChange', 'priceChangePercent', 'lastPrice', 'volume', 'quoteVolume']

    for col in numeric_cols:

        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['openTime'] = pd.to_datetime(df['openTime'], unit='ms')

    return df

clean_df = transform_binance()

print(clean_df.head())



   symbol   priceChange  priceChangePercent  lastPrice     volume  \
0  ETHBTC -2.000000e-04              -0.642   0.030970  21505.082   
1  LTCBTC -8.000000e-06              -1.075   0.000736   7931.036   
2  BNBBTC -4.100000e-05              -0.490   0.008334  14859.257   
3  NEOBTC -9.600000e-07              -2.417   0.000039   2495.960   
4  GASBTC -5.000000e-07              -2.146   0.000023  26217.900   

   quoteVolume                openTime  
0   673.824649 2026-04-17 11:51:41.864  
1     5.848001 2026-04-17 11:51:35.748  
2   123.625672 2026-04-17 11:51:41.749  
3     0.097851 2026-04-17 11:51:27.534  
4     0.603141 2026-04-17 11:51:43.008  


In [5]:
# load to postgres

import psycopg2 as psy 
from sqlalchemy import create_engine
from transform import clean_df
from config import host,port,password,dbname,user
from dotenv import load_dotenv
import os


def load_to_postgres():

    try:

        engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")
                               
        clean_df.to_sql('binance_tickers', con=engine, if_exists='replace', index=False)

        print(f"Successfully loaded {len(clean_df)} rows to the database.")

            
    except Exception as e:
        print(f"Error occurred: {e}")

load_to_postgres()


Error occurred: (psycopg2.OperationalError) could not translate host name "your_host" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [6]:
# run pipeline

from extract import extract_binance
from transform import transform_binance
from load import load_to_postgres
import pandas as pd
import requests
from sqlalchemy import create_engine
from config import host, port, password, dbname, user

# --- 1. EXTRACT ---

def extract_binance():
    print("Step 1: Extracting from Binance...")
    url = "https://api.binance.com/api/v3/ticker/24hr"
    params = {
        'symbolStatus': 'TRADING',
        'type': 'FULL'
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

# --- 2. TRANSFORM ---

def transform_binance(data):

    print("Step 2: Transforming data with Pandas...")

    df = pd.DataFrame(data)
    
    cols = ['symbol', 'priceChange', 'priceChangePercent', 'lastPrice', 'volume', 'openTime']
    df = df[cols]
    
    numeric_cols = ['priceChange', 'priceChangePercent', 'lastPrice', 'volume']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])
        
    df['openTime'] = pd.to_datetime(df['openTime'], unit='ms')
    
    df = df[df['symbol'].str.endswith('USDT')]
    
    return df

# --- 3. LOAD ---

def load_to_postgres(df):

    print(f"Step 3: Loading {len(df)} rows to Aiven Cloud...")

    engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require")
    
    df.to_sql('binance_tickers', con=engine, if_exists='append', index=False)

    print("Success: Pipeline complete.")

# --- RUN THE ETL ---
if __name__ == "__main__":

    try:
        # The Flow
        raw_data = extract_binance()
        clean_df = transform_binance(raw_data)
        load_to_postgres(clean_df)
        
    except Exception as e:
        
        print(f"Pipeline failed! Error: {e}")

Step 1: Extracting from Binance...
Step 2: Transforming data with Pandas...
Step 3: Loading 439 rows to Aiven Cloud...
Pipeline failed! Error: (psycopg2.OperationalError) could not translate host name "your_host" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)
